In [12]:
import pandas as pd
import requests
import json

print("✅ Imports successful")

✅ Imports successful


In [14]:
df = pd.read_csv("online_retail.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 541910
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [89]:
df["Revenue"] = df["Quantity"] * df["Price"]

print("Total Raw Revenue:", df["Revenue"].sum())

Total Raw Revenue: 9747765.934


In [32]:
clean_df = df.copy()

# Identify cancelled invoices
cancelled = clean_df["Invoice"].astype(str).str.startswith("C")

# Remove cancelled invoices
clean_df = clean_df[~cancelled]

# Remove negative-price records
clean_df = clean_df[clean_df["Price"] >= 0]

# Recalculate revenue
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["Price"]

print("Original rows:", len(df))
print("Clean rows:", len(clean_df))
print("Removed rows:", len(df) - len(clean_df))
print("Clean Revenue:", clean_df["Revenue"].sum())

Original rows: 541910
Clean rows: 532620
Removed rows: 9290
Clean Revenue: 10666702.544000005


In [34]:
clean_df["InvoiceDate"] = pd.to_datetime(
    clean_df["InvoiceDate"],
    dayfirst=True
)

print("Start Date:", clean_df["InvoiceDate"].min())
print("End Date:", clean_df["InvoiceDate"].max())

Start Date: 2010-12-01 08:26:00
End Date: 2011-12-09 12:50:00


In [36]:
daily_revenue = (
    clean_df
    .groupby(clean_df["InvoiceDate"].dt.date)["Revenue"]
    .sum()
    .reset_index(name="Revenue")
)

print("Number of Days:", len(daily_revenue))
print(daily_revenue.head())

Number of Days: 305
  InvoiceDate   Revenue
0  2010-12-01  58960.79
1  2010-12-02  47748.38
2  2010-12-03  46943.71
3  2010-12-05  31774.95
4  2010-12-06  54830.46


In [38]:
def detect_revenue_anomalies():
    temp = daily_revenue.copy()

    mean_revenue = temp["Revenue"].mean()
    std_revenue = temp["Revenue"].std()

    temp["z_score"] = (
        (temp["Revenue"] - mean_revenue) / std_revenue
    )

    anomalies = temp[
        temp["z_score"].abs() > 2
    ].copy()

    anomalies = anomalies.sort_values(
        "z_score",
        key=abs,
        ascending=False
    )

    return anomalies.head(10).to_dict(orient="records")


print("Anomaly detection tool ready ✅")

Anomaly detection tool ready ✅


In [95]:
anomalies = detect_revenue_anomalies()

print(json.dumps(anomalies, indent=2, default=str))

[
  {
    "InvoiceDate": "2011-12-09",
    "Revenue": 200938.6,
    "z_score": 7.859188986697585
  },
  {
    "InvoiceDate": "2011-11-14",
    "Revenue": 114419.89,
    "z_score": 3.7621589165445712
  },
  {
    "InvoiceDate": "2011-09-20",
    "Revenue": 109612.03,
    "z_score": 3.5344862317219956
  },
  {
    "InvoiceDate": "2010-12-07",
    "Revenue": 99618.2,
    "z_score": 3.061235740284972
  },
  {
    "InvoiceDate": "2011-01-18",
    "Revenue": 95978.05,
    "z_score": 2.888859106261332
  },
  {
    "InvoiceDate": "2011-12-05",
    "Revenue": 88741.96,
    "z_score": 2.5461993703460193
  },
  {
    "InvoiceDate": "2011-11-07",
    "Revenue": 85881.81,
    "z_score": 2.4107590643688708
  },
  {
    "InvoiceDate": "2011-12-08",
    "Revenue": 82495.0,
    "z_score": 2.250379160277664
  },
  {
    "InvoiceDate": "2011-11-23",
    "Revenue": 80104.18,
    "z_score": 2.1371636323031575
  },
  {
    "InvoiceDate": "2011-09-15",
    "Revenue": 78218.95,
    "z_score": 2.04788994804278

In [97]:
response = requests.get(
    "http://localhost:11434/api/tags",
    timeout=10
)

print("Status:", response.status_code)

if response.status_code == 200:
    print("✅ Ollama is connected")
else:
    print("❌ Ollama connection problem")

Status: 200
✅ Ollama is connected


In [98]:
models = response.json()["models"]

for model in models:
    print(model["name"])

qwen3:4b


In [99]:
anomaly_tool = {
    "type": "function",
    "function": {
        "name": "detect_revenue_anomalies",
        "description": "Finds unusual daily revenue values using statistical z-score analysis and returns the top anomalies.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
}

print("✅ Anomaly tool defined")

✅ Anomaly tool defined


In [103]:
def analyze_revenue_by_country(date):
    date = pd.to_datetime(date).date()

    data = clean_df[
        clean_df["InvoiceDate"].dt.date == date
    ]

    result = (
        data.groupby("Country")["Revenue"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    return result.to_dict(orient="records")


country_tool = {
    "type": "function",
    "function": {
        "name": "analyze_revenue_by_country",
        "description": "Analyzes revenue by country for a specific date and returns the top 10 countries by revenue.",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {
                    "type": "string",
                    "description": "Date in YYYY-MM-DD format."
                }
            },
            "required": ["date"]
        }
    }
}

print("✅ Country analysis tool defined")

✅ Country analysis tool defined


In [105]:
tools = [
    anomaly_tool,
    country_tool
]

print("Available tools:")

for tool in tools:
    print("-", tool["function"]["name"])

Available tools:
- detect_revenue_anomalies
- analyze_revenue_by_country


In [107]:
user_question = """
Investigate the business revenue data.

First identify the most significant revenue anomaly.
Then determine which countries contributed most to that anomaly.
Finally, provide a short business explanation.
"""

print(user_question)


Investigate the business revenue data.

First identify the most significant revenue anomaly.
Then determine which countries contributed most to that anomaly.
Finally, provide a short business explanation.



In [109]:
print("🚀 Sending request to Qwen...")

try:
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "qwen3:4b",
            "messages": [
                {
                    "role": "user",
                    "content": "Find the most significant revenue anomaly."
                }
            ],
            "tools": tools,
            "stream": False
        },
        timeout=120
    )

    print("✅ Response received")
    print("Status:", response.status_code)
    print("Length:", len(response.text))

    result = response.json()

    print("Tool calls:")
    print(result.get("message", {}).get("tool_calls", []))

    print("\nContent:")
    print(result.get("message", {}).get("content", ""))

except Exception as e:
    print("❌ ERROR:", repr(e))

🚀 Sending request to Qwen...
✅ Response received
Status: 200
Length: 2080
Tool calls:
[{'id': 'call_elu9hfa2', 'function': {'index': 0, 'name': 'detect_revenue_anomalies', 'arguments': {}}}]

Content:



In [110]:
simple_tools = [anomaly_tool]

print("🚀 Testing Qwen with ONE tool...")

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": [
            {
                "role": "user",
                "content": "Find the most significant revenue anomaly."
            }
        ],
        "tools": simple_tools,
        "stream": False
    },
    timeout=180
)

print("Status:", response.status_code)

result = response.json()

print(json.dumps(result, indent=2))

🚀 Testing Qwen with ONE tool...
Status: 200
{
  "model": "qwen3:4b",
  "created_at": "2026-08-18T10:36:51.2543512Z",
  "message": {
    "role": "assistant",
    "content": "",
    "thinking": "Okay, let's see. The user wants to find the most significant revenue anomaly. The tool provided is detect_revenue_anomalies, which uses z-score analysis on daily revenue.\n\nFirst, I need to check what parameters this function requires. The description says it takes daily revenue values, but the function signature shows arguments as empty objects or arrays. Wait, the tool's function definition says: \"detect_revenue_anomalies Finds unusual daily revenue values using statistical z-score analysis and returns the top anomalies. {object <nil> <nil> [] {}}\". Hmm, maybe the arguments are not specified here. Wait, the user hasn't provided any data yet. The function might need the revenue data as input.\n\nWait, the problem is that the user hasn't given any specific data. The function might require the 

In [111]:
tool_calls = result["message"]["tool_calls"]

tool_call = tool_calls[0]

tool_name = tool_call["function"]["name"]
arguments = tool_call["function"]["arguments"]

print("Tool selected:", tool_name)
print("Arguments:", arguments)

if tool_name == "detect_revenue_anomalies":
    tool_result = detect_revenue_anomalies()

print("\nTool result:")
print(json.dumps(tool_result, indent=2, default=str))

Tool selected: detect_revenue_anomalies
Arguments: {}

Tool result:
[
  {
    "InvoiceDate": "2011-12-09",
    "Revenue": 200938.6,
    "z_score": 7.859188986697585
  },
  {
    "InvoiceDate": "2011-11-14",
    "Revenue": 114419.89,
    "z_score": 3.7621589165445712
  },
  {
    "InvoiceDate": "2011-09-20",
    "Revenue": 109612.03,
    "z_score": 3.5344862317219956
  },
  {
    "InvoiceDate": "2010-12-07",
    "Revenue": 99618.2,
    "z_score": 3.061235740284972
  },
  {
    "InvoiceDate": "2011-01-18",
    "Revenue": 95978.05,
    "z_score": 2.888859106261332
  },
  {
    "InvoiceDate": "2011-12-05",
    "Revenue": 88741.96,
    "z_score": 2.5461993703460193
  },
  {
    "InvoiceDate": "2011-11-07",
    "Revenue": 85881.81,
    "z_score": 2.4107590643688708
  },
  {
    "InvoiceDate": "2011-12-08",
    "Revenue": 82495.0,
    "z_score": 2.250379160277664
  },
  {
    "InvoiceDate": "2011-11-23",
    "Revenue": 80104.18,
    "z_score": 2.1371636323031575
  },
  {
    "InvoiceDate": "2

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Find the most significant revenue anomaly."
    },
    {
        "role": "assistant",
        "tool_calls": tool_calls
    },
    {
        "role": "tool",
        "content": json.dumps(tool_result, default=str)
    }
]

print("Sending tool result back to Qwen...")

final_response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": messages,
        "stream": False
    },
    timeout=120
)

final_result = final_response.json()

print(final_result["message"]["content"])

In [113]:
def analyze_revenue_by_country(date):
    date = pd.to_datetime(date).date()

    data = clean_df[
        clean_df["InvoiceDate"].dt.date == date
    ]

    result = (
        data.groupby("Country")["Revenue"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    return result.to_dict(orient="records")


print("✅ Country analysis function ready")

✅ Country analysis function ready


In [114]:
country_tool = {
    "type": "function",
    "function": {
        "name": "analyze_revenue_by_country",
        "description": "Analyzes revenue by country for a specific date and returns the top 10 countries by revenue.",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {
                    "type": "string",
                    "description": "Date to analyze in YYYY-MM-DD format."
                }
            },
            "required": ["date"]
        }
    }
}

print("✅ Tool 2 registered:", country_tool["function"]["name"])

✅ Tool 2 registered: analyze_revenue_by_country


In [40]:
country_result = analyze_revenue_by_country("2011-12-09")

print(json.dumps(country_result, indent=2))

[
  {
    "Country": "United Kingdom",
    "Revenue": 196134.1
  },
  {
    "Country": "Norway",
    "Revenue": 2638.69
  },
  {
    "Country": "Germany",
    "Revenue": 1689.72
  },
  {
    "Country": "France",
    "Revenue": 267.45
  },
  {
    "Country": "Belgium",
    "Revenue": 208.64000000000001
  }
]


In [119]:
clean_df["InvoiceDate"] = pd.to_datetime(
    clean_df["InvoiceDate"],
    dayfirst=True,
    errors="coerce"
)

print(clean_df["InvoiceDate"].dtype)
print(clean_df["InvoiceDate"].min())
print(clean_df["InvoiceDate"].max())

datetime64[ns]
2010-12-01 08:26:00
2011-12-09 12:50:00


In [121]:
country_result = analyze_revenue_by_country("2011-12-09")

print(json.dumps(country_result, indent=2))

[
  {
    "Country": "United Kingdom",
    "Revenue": 196134.1
  },
  {
    "Country": "Norway",
    "Revenue": 2638.69
  },
  {
    "Country": "Germany",
    "Revenue": 1689.72
  },
  {
    "Country": "France",
    "Revenue": 267.45
  },
  {
    "Country": "Belgium",
    "Revenue": 208.64000000000001
  }
]


In [123]:
tools = [
    anomaly_tool,
    country_tool
]

print("Tools available to Qwen:")

for tool in tools:
    print("→", tool["function"]["name"])

Tools available to Qwen:
→ detect_revenue_anomalies
→ analyze_revenue_by_country


In [44]:
user_question = """
Investigate the revenue data.

Find the most significant revenue anomaly.
After identifying the anomaly date, analyze which countries
contributed the most revenue on that date.

Return a concise business explanation.
"""

print("Investigation task ready ✅")

Investigation task ready ✅


In [ ]:
tool_calls = result["message"].get("tool_calls", [])

tool_call = tool_calls[0]

tool_name = tool_call["function"]["name"]
arguments = tool_call["function"].get("arguments", {})

print("Tool selected:", tool_name)
print("Arguments:", arguments)

if tool_name == "detect_revenue_anomalies":
    tool_result = detect_revenue_anomalies()

elif tool_name == "analyze_revenue_by_country":
    tool_result = analyze_revenue_by_country(arguments["date"])

print("\nTool result:")
print(json.dumps(tool_result, indent=2, default=str))

In [55]:
messages = [
    {
        "role": "user",
        "content": user_question
    },
    {
        "role": "assistant",
        "tool_calls": tool_calls
    },
    {
        "role": "tool",
        "content": json.dumps(tool_result, default=str)
    }
]

print("🚀 Sending anomaly result back to Qwen...")

response2 = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": messages,
        "tools": tools,
        "stream": False
    },
    timeout=180
)

result2 = response2.json()

print("Status:", response2.status_code)
print("\nQwen response:")
print(json.dumps(result2["message"], indent=2))

🚀 Sending anomaly result back to Qwen...
Status: 200

Qwen response:
{
  "role": "assistant",
  "content": "",
  "thinking": "Okay, let's tackle this problem step by step. The user wants to investigate the revenue data, find the most significant revenue anomaly, and then analyze which countries contributed the most revenue on that anomaly date.\n\nFirst, I called the detect_revenue_anomalies function. The response gave me a list of dates with their revenue and z-scores. The z-score is a measure of how many standard deviations an element is from the mean. The higher the z-score, the more unusual the value. Looking at the data, the top anomaly here is the date \"2011-12-09\" with a z-score of 7.859, which is way higher than the others. That's definitely the most significant anomaly.\n\nNext, the user wants to know which countries contributed the most revenue on that date. So I need to use the analyze_revenue_by_country function. But wait, the function requires a date. The anomaly date we

In [57]:
tool_calls_2 = result2["message"].get("tool_calls", [])

tool_call_2 = tool_calls_2[0]

tool_name_2 = tool_call_2["function"]["name"]
arguments_2 = tool_call_2["function"]["arguments"]

print("Tool selected:", tool_name_2)
print("Arguments:", arguments_2)

if tool_name_2 == "analyze_revenue_by_country":
    country_result = analyze_revenue_by_country(
        arguments_2["date"]
    )

print("\nCountry analysis:")
print(json.dumps(country_result, indent=2))

Tool selected: analyze_revenue_by_country
Arguments: {'date': '2011-12-09'}

Country analysis:
[
  {
    "Country": "United Kingdom",
    "Revenue": 196134.1
  },
  {
    "Country": "Norway",
    "Revenue": 2638.69
  },
  {
    "Country": "Germany",
    "Revenue": 1689.72
  },
  {
    "Country": "France",
    "Revenue": 267.45
  },
  {
    "Country": "Belgium",
    "Revenue": 208.64000000000001
  }
]


In [ ]:
messages_2 = [
    {
        "role": "user",
        "content": user_question
    },
    {
        "role": "assistant",
        "tool_calls": tool_calls
    },
    {
        "role": "tool",
        "content": json.dumps(tool_result, default=str)
    },
    {
        "role": "assistant",
        "tool_calls": tool_calls_2
    },
    {
        "role": "tool",
        "content": json.dumps(country_result, default=str)
    }
]

print("🚀 Asking Qwen for final business insight...")

final_response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": messages_2,
        "stream": False
    },
    timeout=180
)

final_result = final_response.json()

print("\nFINAL BUSINESS INSIGHT:\n")
print(final_result["message"]["content"])

In [61]:
def analyze_revenue_by_product(date, country=None):
    date = pd.to_datetime(date).date()

    data = clean_df[
        clean_df["InvoiceDate"].dt.date == date
    ].copy()

    if country:
        data = data[data["Country"] == country]

    result = (
        data.groupby(["StockCode", "Description"])["Revenue"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    return result.to_dict(orient="records")


print("✅ Product analysis function ready")

✅ Product analysis function ready


In [ ]:
product_result = analyze_revenue_by_product(
    "2011-12-09",
    "United Kingdom"
)

print(json.dumps(product_result, indent=2, default=str))


In [65]:
product_tool = {
    "type": "function",
    "function": {
        "name": "analyze_revenue_by_product",
        "description": "Analyzes the top revenue-generating products for a specific date and optionally a specific country.",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {
                    "type": "string",
                    "description": "Date in YYYY-MM-DD format."
                },
                "country": {
                    "type": "string",
                    "description": "Optional country to filter."
                }
            },
            "required": ["date"]
        }
    }
}

print("✅ Tool 3 registered")

✅ Tool 3 registered


In [ ]:
tools = [
    anomaly_tool,
    country_tool,
    product_tool
]

print("Available tools:")

for tool in tools:
    print("→", tool["function"]["name"])

In [69]:
print("Top products on 2011-12-09 in United Kingdom:\n")

for item in product_result:
    print(
        item["StockCode"],
        "|",
        item["Description"],
        "| £",
        round(item["Revenue"], 2)
    )

Top products on 2011-12-09 in United Kingdom:

23843 | PAPER CRAFT , LITTLE BIRDIE | £ 168469.6
DOT | DOTCOM POSTAGE | £ 2647.34
21137 | BLACK RECORD COVER FRAME | £ 669.57
22086 | PAPER CHAIN KIT 50'S CHRISTMAS  | £ 591.9
20749 | ASSORTED COLOUR MINI CASES | £ 533.4
23084 | RABBIT NIGHT LIGHT | £ 528.45
22114 | HOT WATER BOTTLE TEA AND SYMPATHY | £ 504.56
23404 | HOME SWEET HOME BLACKBOARD | £ 469.44
23355 | HOT WATER BOTTLE KEEP CALM | £ 437.04
22423 | REGENCY CAKESTAND 3 TIER | £ 327.18


In [71]:
tools = [
    anomaly_tool,
    country_tool,
    product_tool
]

print("Agent tools ready ✅")

for tool in tools:
    print("→", tool["function"]["name"])

Agent tools ready ✅
→ detect_revenue_anomalies
→ analyze_revenue_by_country
→ analyze_revenue_by_product


In [ ]:
agent_question = """
Investigate the business revenue anomaly.

1. Find the most significant revenue anomaly.
2. Identify the country contributing the most revenue on that date.
3. Identify the top products contributing to that country's revenue.
4. Explain the likely business driver of the anomaly.
5. Use GBP (£) as the currency.

Provide a concise business investigation report.
"""

print("Full investigation task ready ✅")

In [ ]:
import requests

r = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": [
            {"role": "user", "content": "Say OK"}
        ],
        "stream": False
    },
    timeout=60
)

print(r.status_code)
print(r.json()["message"]["content"])

In [2]:
import pandas as pd
import numpy as np
import json
import requests

print("✅ Kernel working")

✅ Kernel working


In [8]:
def analyze_revenue_by_country(date):
    date = pd.to_datetime(date).date()

    data = clean_df[
        clean_df["InvoiceDate"].dt.date == date
    ]

    result = (
        data.groupby("Country")["Revenue"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    return result.to_dict(orient="records")


print("✅ Country analysis function restored")

✅ Country analysis function restored


In [48]:
country_result = analyze_revenue_by_country("2011-12-09")

print(json.dumps(country_result, indent=2))

[
  {
    "Country": "United Kingdom",
    "Revenue": 196134.1
  },
  {
    "Country": "Norway",
    "Revenue": 2638.69
  },
  {
    "Country": "Germany",
    "Revenue": 1689.72
  },
  {
    "Country": "France",
    "Revenue": 267.45
  },
  {
    "Country": "Belgium",
    "Revenue": 208.64000000000001
  }
]


In [50]:
import pandas as pd
import numpy as np
import json
import requests

file_path = r"online_retail.csv"

df = pd.read_csv(file_path, encoding="ISO-8859-1")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())


Rows: 541910
Columns: ['ï»¿Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [66]:
clean_df = df.copy()

# Convert date
clean_df["InvoiceDate"] = pd.to_datetime(
    clean_df["InvoiceDate"],
    dayfirst=True,
    errors="coerce"
)

# Revenue
clean_df["Revenue"] = (
    clean_df["Quantity"] * clean_df["Price"]
)

# Remove cancelled invoices
clean_df = clean_df[
    ~clean_df["Invoice"].astype(str).str.startswith("C")
].copy()

print("Clean rows:", len(clean_df))
print("Revenue:", round(clean_df["Revenue"].sum(), 2))
print("Date type:", clean_df["InvoiceDate"].dtype)

Clean rows: 532622
Revenue: 10644578.42
Date type: datetime64[ns]


In [28]:
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (541910, 8)
Columns:
['ï»¿Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [64]:
df.rename(columns={"ï»¿Invoice": "Invoice"}, inplace=True)

print(df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [70]:
anomaly_tool = {
    "type": "function",
    "function": {
        "name": "detect_revenue_anomalies",
        "description": "Finds unusual daily revenue values using statistical z-score analysis. Returns the top revenue anomalies.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
}

print("✅ Anomaly tool restored")

✅ Anomaly tool restored


In [72]:
print("🚀 Starting Agent...")

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": [
            {
                "role": "user",
                "content": """
Find the most significant revenue anomaly.
Return the anomaly date and explain why it is significant.
"""
            }
        ],
        "tools": [anomaly_tool],
        "stream": False
    },
    timeout=120
)

result = response.json()

print("Status:", response.status_code)
print(json.dumps(result["message"], indent=2))

🚀 Starting Agent...
Status: 200
{
  "role": "assistant",
  "content": "",
  "thinking": "Okay, let me try to figure out how to handle this user query. The user wants to find the most significant revenue anomaly, specifically the anomaly date and an explanation of why it's significant. \n\nFirst, I need to check the tools provided. The only tool available is detect_revenue_anomalies, which uses z-score analysis on daily revenue to find unusual values and returns the top anomalies. \n\nWait, the function's description says it \"Finds unusual daily revenue values using statistical z-score analysis. Returns the top revenue anomalies.\" So the tool should give me the top anomalies, including their dates and maybe why they're significant.\n\nBut the user is asking for the most significant one. The function returns the top anomalies, so I probably need to call this function with some arguments. Wait, the arguments section in the tool says: {object <nil> <nil> [] {}}, which is a bit confusing.

In [78]:
country_tool = {
    "type": "function",
    "function": {
        "name": "analyze_revenue_by_country",
        "description": "Analyzes revenue by country for a specific date and returns the top 10 countries by revenue.",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {
                    "type": "string",
                    "description": "Date to analyze in YYYY-MM-DD format."
                }
            },
            "required": ["date"]
        }
    }
}

messages = [
    {
        "role": "user",
        "content": "Find the most significant revenue anomaly and then investigate which countries contributed most on that date."
    },
    result["message"],
    {
        "role": "tool",
        "tool_call_id": result["message"]["tool_calls"][0]["id"],
        "content": json.dumps(tool_result)
    }
]

print("🚀 Sending anomaly result back to Qwen...")

response2 = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": messages,
        "tools": [country_tool],
        "stream": False
    },
    timeout=120
)

result2 = response2.json()

print("Status:", response2.status_code)
print(json.dumps(result2["message"], indent=2))

🚀 Sending anomaly result back to Qwen...
Status: 200
{
  "role": "assistant",
  "content": "",
  "thinking": "Okay, let's see. The user asked for the most significant revenue anomaly and then to investigate which countries contributed most on that date.\n\nFirst, I called the detect_revenue_anomalies function, which returned three anomalies. The top one is from 2011-12-09 with a z-score of 7.859, which is way higher than the others. That's the most significant anomaly.\n\nNow, the next step is to find which countries contributed most on that date. The tool provided earlier (analyze_revenue_by_country) takes a date and returns top 10 countries by revenue. So I need to use that function with the date 2011-12-09.\n\nWait, the user's original query says \"investigate which countries contributed most on that date\". The \"that date\" refers to the most significant anomaly date, which is 2011-12-09.\n\nSo I should call analyze_revenue_by_country with date \"2011-12-09\".\n\nLet me check the 

In [80]:
country_result = [
    {"Country": "United Kingdom", "Revenue": 196134.10},
    {"Country": "Norway", "Revenue": 2638.69},
    {"Country": "Germany", "Revenue": 1689.72},
    {"Country": "France", "Revenue": 267.45},
    {"Country": "Belgium", "Revenue": 208.64}
]

messages2 = [
    {
        "role": "user",
        "content": "Find the most significant revenue anomaly and investigate which countries contributed most on that date. Give a concise final business explanation."
    },
    result["message"],
    {
        "role": "tool",
        "tool_call_id": result["message"]["tool_calls"][0]["id"],
        "content": json.dumps(tool_result)
    },
    result2["message"],
    {
        "role": "tool",
        "tool_call_id": result2["message"]["tool_calls"][0]["id"],
        "content": json.dumps(country_result)
    }
]

print("🚀 Sending country result to Qwen for final insight...")

final_response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": messages2,
        "stream": False
    },
    timeout=120
)

final_result = final_response.json()

print("Status:", final_response.status_code)
print("\nFINAL QWEN INSIGHT:\n")
print(final_result["message"]["content"])

🚀 Sending country result to Qwen for final insight...
Status: 200

FINAL QWEN INSIGHT:

The most significant revenue anomaly occurred on **2011-12-09** (z-score: 7.86), representing a massive outlier in daily revenue. On this date, the **United Kingdom** contributed **$196,134.10**—over 98% of the anomaly's total revenue—followed by Norway ($2,638.69) and Germany ($1,689.72). 

**Business Explanation**: This extreme spike in UK revenue likely resulted from a high-value transaction (e.g., a major client order, seasonal surge, or data error). Given the exceptionally high z-score (7.86), it represents a statistically significant deviation from normal patterns. Immediate investigation is critical to determine whether this was a genuine business opportunity, a system error, or an outlier requiring corrective action. The UK’s dominant contribution suggests targeting this region for future revenue strategies or process optimization.


In [82]:
product_tool = {
    "type": "function",
    "function": {
        "name": "analyze_top_products",
        "description": "Finds the top products by revenue for a specific date and country.",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {
                    "type": "string",
                    "description": "Date in YYYY-MM-DD format."
                },
                "country": {
                    "type": "string",
                    "description": "Country to analyze."
                }
            },
            "required": ["date", "country"]
        }
    }
}

print("✅ Product analysis tool ready")

✅ Product analysis tool ready


In [84]:
def analyze_top_products(date, country):
    date = pd.to_datetime(date).date()

    data = clean_df[
        (clean_df["InvoiceDate"].dt.date == date) &
        (clean_df["Country"] == country)
    ]

    result = (
        data.groupby(["StockCode", "Description"])["Revenue"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    return result.to_dict(orient="records")


print("✅ Product analysis function ready")

✅ Product analysis function ready


In [86]:
product_result = analyze_top_products(
    "2011-12-09",
    "United Kingdom"
)

print(json.dumps(product_result, indent=2))

[
  {
    "StockCode": "23843",
    "Description": "PAPER CRAFT , LITTLE BIRDIE",
    "Revenue": 168469.6
  },
  {
    "StockCode": "DOT",
    "Description": "DOTCOM POSTAGE",
    "Revenue": 2647.34
  },
  {
    "StockCode": "21137",
    "Description": "BLACK RECORD COVER FRAME",
    "Revenue": 669.57
  },
  {
    "StockCode": "22086",
    "Description": "PAPER CHAIN KIT 50'S CHRISTMAS ",
    "Revenue": 591.9
  },
  {
    "StockCode": "20749",
    "Description": "ASSORTED COLOUR MINI CASES",
    "Revenue": 533.4
  },
  {
    "StockCode": "23084",
    "Description": "RABBIT NIGHT LIGHT",
    "Revenue": 528.45
  },
  {
    "StockCode": "22114",
    "Description": "HOT WATER BOTTLE TEA AND SYMPATHY",
    "Revenue": 504.55999999999995
  },
  {
    "StockCode": "23404",
    "Description": "HOME SWEET HOME BLACKBOARD",
    "Revenue": 469.43999999999994
  },
  {
    "StockCode": "23355",
    "Description": "HOT WATER BOTTLE KEEP CALM",
    "Revenue": 437.0399999999999
  },
  {
    "StockCode"

In [92]:
print("🔥 FINAL BUSINESS INVESTIGATION")
print("=" * 55)

total_revenue = 200938.60
uk_revenue = 196134.10
top_product = 168469.60

uk_share = (uk_revenue / total_revenue) * 100
product_share = (top_product / uk_revenue) * 100

print(f"""
ANOMALY
Date: 2011-12-09
Revenue: £{total_revenue:,.2f}
Z-score: 7.86

COUNTRY DRIVER
United Kingdom: £{uk_revenue:,.2f}
Contribution: {uk_share:.2f}%

PRODUCT DRIVER
PAPER CRAFT, LITTLE BIRDIE: £{top_product:,.2f}
Share of UK revenue: {product_share:.2f}%

BUSINESS INSIGHT
The revenue spike on 2011-12-09 is an extreme statistical anomaly.
The United Kingdom generated approximately {uk_share:.1f}% of the day's revenue,
making it the dominant market behind the spike.

Within the UK, PAPER CRAFT, LITTLE BIRDIE generated approximately
£{top_product:,.2f}, accounting for {product_share:.1f}% of UK revenue.

RECOMMENDED ACTION
Investigate the underlying transactions for this product and date.
Check whether the spike represents genuine bulk demand, a seasonal event,
duplicate transactions, pricing issues, or a data-quality problem.

AGENT WORKFLOW
Anomaly Detection → Country Attribution → Product Attribution → Business Recommendation

STATUS: PROJECT COMPLETE ✅
""")

🔥 FINAL BUSINESS INVESTIGATION

ANOMALY
Date: 2011-12-09
Revenue: £200,938.60
Z-score: 7.86

COUNTRY DRIVER
United Kingdom: £196,134.10
Contribution: 97.61%

PRODUCT DRIVER
PAPER CRAFT, LITTLE BIRDIE: £168,469.60
Share of UK revenue: 85.90%

BUSINESS INSIGHT
The revenue spike on 2011-12-09 is an extreme statistical anomaly.
The United Kingdom generated approximately 97.6% of the day's revenue,
making it the dominant market behind the spike.

Within the UK, PAPER CRAFT, LITTLE BIRDIE generated approximately
£168,469.60, accounting for 85.9% of UK revenue.

RECOMMENDED ACTION
Investigate the underlying transactions for this product and date.
Check whether the spike represents genuine bulk demand, a seasonal event,
duplicate transactions, pricing issues, or a data-quality problem.

AGENT WORKFLOW
Anomaly Detection → Country Attribution → Product Attribution → Business Recommendation

STATUS: PROJECT COMPLETE ✅



In [96]:
# CELL 42 — Transaction-level validation

invoice_col = next(
    col for col in clean_df.columns
    if "invoice" in col.lower()
)

target = clean_df[
    (pd.to_datetime(clean_df["InvoiceDate"]).dt.date == pd.to_datetime("2011-12-09").date()) &
    (clean_df["Country"] == "United Kingdom") &
    (clean_df["Description"] == "PAPER CRAFT , LITTLE BIRDIE")
].copy()

print("🔎 TRANSACTION-LEVEL VALIDATION")
print("=" * 55)

print("Invoice column:", invoice_col)
print("Rows:", len(target))
print("Total Quantity:", target["Quantity"].sum())
print("Average Price:", round(target["Price"].mean(), 2))
print("Total Revenue:", round(target["Revenue"].sum(), 2))
print("Unique Invoices:", target[invoice_col].nunique())

print("\nTop invoices:")
print(
    target.groupby(invoice_col)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

🔎 TRANSACTION-LEVEL VALIDATION
Invoice column: Invoice
Rows: 1
Total Quantity: 80995
Average Price: 2.08
Total Revenue: 168469.6
Unique Invoices: 1

Top invoices:
Invoice
581483    168469.6
Name: Revenue, dtype: float64
